In [14]:
#Importing needed libraries and texts
import nltk
from nltk.corpus import gutenberg

import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
import time

random.seed(123)
np.random.seed(123)
tf.random.set_seed(123)

nltk.download('gutenberg')

author = "Jane Austen"
file_ids = ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt']
titles = ["Emma", "Persuasion", "Sense and Sensibility"]

combined_text = ""

for file_id, title in zip(file_ids, titles):
    text = gutenberg.raw(file_id)
    combined_text += text
    print(f"Title: {title}")
    print(f"File ID: {file_id}")
    print(f"Total Characters: {len(text):,}")
    print(f"Unique Characters: {len(set(text))}")
    print("-" * 40)

print(f"Author: {author}")
print(f"Combined Total Characters: {len(combined_text):,}")
print(f"Combined Unique Characters: {len(set(combined_text))}")

Title: Emma
File ID: austen-emma.txt
Total Characters: 887,071
Unique Characters: 77
----------------------------------------
Title: Persuasion
File ID: austen-persuasion.txt
Total Characters: 466,292
Unique Characters: 78
----------------------------------------
Title: Sense and Sensibility
File ID: austen-sense.txt
Total Characters: 673,022
Unique Characters: 78
----------------------------------------
Author: Jane Austen
Combined Total Characters: 2,026,385
Combined Unique Characters: 82


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [15]:
#Creating character-to-index and index-to-character mappings
chars = sorted(list(set(combined_text)))

char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}

print(f"Total unique characters mapped: {len(chars)}")
print(f"Sample char_to_index: {list(char_to_index.items())[:5]}")
print(f"Sample index_to_char: {list(index_to_char.items())[:5]}")

Total unique characters mapped: 82
Sample char_to_index: [('\n', 0), (' ', 1), ('!', 2), ('"', 3), ('&', 4)]
Sample index_to_char: [(0, '\n'), (1, ' '), (2, '!'), (3, '"'), (4, '&')]


In [16]:
#Preparing overlapping input sequences and target characters
seq_length = 40
input_sequences = []
target_chars = []

for i in range(0, len(combined_text) - seq_length):
    input_seq = combined_text[i:i + seq_length]
    target_char = combined_text[i + seq_length]
    input_sequences.append([char_to_index[char] for char in input_seq])
    target_chars.append(char_to_index[target_char])

print(f"Total number of training sequences: {len(input_sequences):,}")
print(f"Sample input sequence (length {seq_length}): {input_sequences[0]}")
print(f"Sample target character index: {target_chars[0]}")

Total number of training sequences: 2,026,345
Sample input sequence (length 40): [52, 30, 68, 68, 56, 1, 57, 80, 1, 35, 56, 69, 60, 1, 26, 76, 74, 75, 60, 69, 1, 13, 20, 13, 18, 53, 0, 0, 47, 40, 37, 46, 38, 30, 1, 34, 0, 0, 28, 33]
Sample target character index: 26


In [17]:
#Building a character-level text generation model using LSTM
num_unique_chars = len(chars)

X = np.array(input_sequences)
y = np.array(target_chars)

model = Sequential([
    Embedding(input_dim=num_unique_chars, output_dim=128),
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(256),
    Dropout(0.2),
    Dense(num_unique_chars, activation='softmax')
])

model.build((None, seq_length))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 40, 128)        │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 40, 256)        │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 82)             │        21,074 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 951,122 (3.63 MB)

 Trainable params: 951,122 (3.63 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
#Training the LSTM model
start_time = time.time()

history = model.fit(X, y, batch_size=128, epochs=10)

end_time = time.time()
training_time = end_time - start_time

print(f"Total Training Time: {training_time:.2f} seconds ({training_time / 60:.2f} minutes)")

Epoch 1/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 266s 17ms/step - accuracy: 0.5426 - loss: 1.5336
Epoch 2/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 321s 17ms/step - accuracy: 0.6086 - loss: 1.2771
Epoch 3/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 263s 17ms/step - accuracy: 0.6207 - loss: 1.2305
Epoch 4/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 263s 17ms/step - accuracy: 0.6259 - loss: 1.2083
Epoch 5/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 263s 17ms/step - accuracy: 0.6294 - loss: 1.1954
Epoch 6/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 262s 17ms/step - accuracy: 0.6312 - loss: 1.1866
Epoch 7/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 263s 17ms/step - accuracy: 0.6326 - loss: 1.1812
Epoch 8/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 263s 17ms/step - accuracy: 0.6339 - loss: 1.1781
Epoch 9/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 263s 17ms/step - accuracy: 0.6344 - loss: 1.1749
Epoch 10/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 263s 17ms/step - accuracy: 0.6348 - loss: 1.1725
Total Training Time: 2691.83 seconds (44.86 minutes)


In [19]:
#Generating three text samples using the trained model and three different sampling temperatures
def generate_text(model, start_string, num_generate=300, temperature=1.0,
                  seq_length=40, enforce_length=True):
    if enforce_length:
        assert len(start_string) == seq_length, \
            f"Seed must be exactly {seq_length} chars, got {len(start_string)}"

    input_eval = [char_to_index[s] for s in start_string]
    text_generated = []

    for _ in range(num_generate):
        eval_tensor = tf.expand_dims(input_eval, 0)
        probs = model(eval_tensor, training=False)[0]

        logits = tf.math.log(probs + 1e-8) / temperature
        predicted_id = int(tf.random.categorical(logits[None, :], num_samples=1)[0, 0])

        text_generated.append(index_to_char[predicted_id])
        input_eval.append(predicted_id)
        input_eval = input_eval[1:]

    return start_string + ''.join(text_generated)


short_seed = "It is a truth universally"
print(f"Short seed length: {len(short_seed)}")

full_seed = "It is a truth universally acknowledged, "
print(f"Full seed length: {len(full_seed)}")

long_seed = "It is a truth universally acknowledged, that a single man in possession of a good fortune, "
print(f"Long seed length: {len(long_seed)}")

for temp in [0.2, 0.7, 1.2]:
    print(f"\n=== Temperature: {temp} ===")

    tf.random.set_seed(123)
    print(f"\n[25-char seed]\n{generate_text(model, short_seed, 300, temp, enforce_length=False)}")

    tf.random.set_seed(123)
    print(f"\n[40-char seed]\n{generate_text(model, full_seed, 300, temp)}")

    tf.random.set_seed(123)
    print(f"\n[91-char seed]\n{generate_text(model, long_seed, 300, temp, enforce_length=False)}")

Short seed length: 25
Full seed length: 40
Long seed length: 91

=== Temperature: 0.2 ===

[25-char seed]
It is a truth universally as they were at the same affection of the strength of the same and some sight of the same time to the same subject of the consideration of the same company, and the subject of the consideration of the same sight of his sister, there was a great deal of his account of the party, and a sort of the co

[40-char seed]
It is a truth universally acknowledged, and the consideration of the same secret of the party, and the more consideration of the same words, and a secret of the same consideration of the consideration of the same words, and she had been a little to be allowed to be so much of the same consideration of the consideration of the time of the 

[91-char seed]
It is a truth universally acknowledged, that a single man in possession of a good fortune, and the consideration of the same secret of the party of the party of the same and assistance of the sam